In [7]:
import pandas as pd

# ================== 工具函数 ==================
# 颜色渲染：正数红色，其余默认
def colorize(val: float, width=10):
    if pd.isna(val):
        return " " * width
    s = f"{val:.3f}".rjust(width)  # 固定宽度，保证对齐
    if val > 0:
        return f"\033[91m{s}\033[0m"  # 红色
    return s

# 表格打印（对齐 + 颜色）
def print_colored_table(df: pd.DataFrame, title: str):
    print(title)
    # 打印列名
    header = " " * 10 + "".join(c.rjust(10) for c in df.columns)
    print(header)
    # 打印每行
    for idx, row in df.iterrows():
        row_str = str(idx).ljust(10)
        for val in row:
            row_str += colorize(val, width=10)
        print(row_str)
    print()


# ================== 主逻辑 ==================
df = pd.read_csv("./history_results/results8/result_summary.csv")
# df = pd.read_csv("./results/result_summary.csv")

loss_list = [f"loss{i}" for i in range(1, 6)]
# loss_list = [f"loss{i}" for i in [3,5]]
# topk_list = ["Top 3", "Top 5", "Top 10", "Top 20"]
topk_list = ["Top 10"]

# ========== 生成表 (平均提升百分比) ==========
def make_tables(df, label):
    for topk in topk_list:
        df_topk = df[df["TopK"] == topk]
        pivot = (
            df_topk[df_topk["Loss"].isin(loss_list)]
            .groupby(["Model", "Loss"])["RelDiff(%)"]
            .mean()
            .reset_index()
        )
        table = pivot.pivot(index="Loss", columns="Model", values="RelDiff(%)").round(3)
        print_colored_table(table, f"\n=== {label} | {topk} ===")


print("========== 汇总表 ==========")

# 遍历 campus
for campus, df_c in df.groupby("Campus"):
    print(f"\n########## Campus: {campus} ##########")
    make_tables(df_c, f"Results | Campus={campus}")


========== 汇总表 ==========

########## Campus: campus_10 ##########

=== Results | Campus=campus_10 | Top 10 ===
                 NCL       SGL    SimGCL   XSimGCL
loss1         -1.604    -2.191    -0.949    -0.606
loss2         -0.871    -1.043     0.422    -0.401
loss3          0.556     0.377     0.406     0.108
loss4         -3.501    -4.212    -2.096    -2.340
loss5          0.356    -0.045     0.604     0.115


########## Campus: campus_102 ##########

=== Results | Campus=campus_102 | Top 10 ===
                 NCL       SGL    SimGCL   XSimGCL
loss1         -3.434    -2.575    -1.861    -2.272
loss2         -0.658    -0.560     0.277    -0.056
loss3          0.368     0.672     1.005    -0.138
loss4         -3.304    -2.695    -1.073    -2.409
loss5          0.285     0.369     0.180    -0.368


########## Campus: campus_143 ##########

=== Results | Campus=campus_143 | Top 10 ===
                 NCL       SGL    SimGCL   XSimGCL
loss1         -1.053    -0.013     1.068     1.

In [8]:
import pandas as pd

# ================== 工具函数 ==================
# 颜色渲染：正数红色，其余默认
def colorize(val: float, width=10):
    if pd.isna(val):
        return " " * width
    s = f"{val:.3f}".rjust(width)  # 固定宽度，保证对齐
    if val > 0:
        return f"\033[91m{s}\033[0m"  # 红色
    return s

# 表格打印（按列宽对齐 + 颜色）
def print_colored_table(df: pd.DataFrame, title: str, value_width: int = 10):
    print(title)
    # 动态确定行名列宽
    row_label_width = max(12, max((len(str(i)) for i in df.index), default=0) + 2)
    # 打印列名
    header = " " * row_label_width + "".join(str(c).rjust(value_width) for c in df.columns)
    print(header)
    # 打印每行
    for idx, row in df.iterrows():
        row_str = str(idx).ljust(row_label_width)
        for val in row:
            row_str += colorize(val, width=value_width)
        print(row_str)
    print()


# ================== 主逻辑 ==================
df = pd.read_csv("./history_results/results8/result_summary.csv")
# df = pd.read_csv("./results/result_summary.csv")

# 确保数值列为数值类型
df["RelDiff(%)"] = pd.to_numeric(df["RelDiff(%)"], errors="coerce")

# 只考虑 loss3 和 loss5
loss_keep = ["loss3", "loss5"]
# topk_list = ["Top 3", "Top 5", "Top 10", "Top 20"]
topk_list = ["Top 3", "Top 5", "Top 10"]

print("========== 汇总表 ==========")

# 过滤数据
subset = df[df["Loss"].isin(loss_keep)]

# 遍历 (Campus, Loss, TopK)，分别打印表格
for campus, df_c in subset.groupby("Campus", sort=True):
    print(f"\n########## Campus: {campus} ##########")
    for loss in loss_keep:
        print(f"###### Loss: {loss} ######")
        df_l = df_c[df_c["Loss"] == loss]
        for topk in topk_list:
            g = df_l[df_l["TopK"] == topk]
            if g.empty:
                continue
            pivot = (
                g.groupby(["Metric", "Model"])["RelDiff(%)"]
                 .mean()
                 .reset_index()
                 .pivot(index="Metric", columns="Model", values="RelDiff(%)")
                 .sort_index()
                 .round(3)
            )
            pivot = pivot.reindex(sorted(pivot.columns), axis=1)  # 模型列排序
            print_colored_table(
                pivot,
                f"=== Results | Campus={campus} | Loss={loss} | TopK={topk} ==="
            )


========== 汇总表 ==========

########## Campus: campus_10 ##########
###### Loss: loss3 ######
=== Results | Campus=campus_10 | Loss=loss3 | TopK=Top 3 ===
                   NCL       SGL    SimGCL   XSimGCL
Hit Ratio        0.679     2.340    -1.369     0.493
NDCG             1.092     1.249    -0.606     0.814
Precision        0.682     2.340    -1.368     0.495
Recall           0.843     1.148    -1.277     0.013

=== Results | Campus=campus_10 | Loss=loss3 | TopK=Top 5 ===
                   NCL       SGL    SimGCL   XSimGCL
Hit Ratio        0.026     1.099    -0.357     0.216
NDCG             0.758     0.563     0.036     0.577
Precision        0.022     1.097    -0.357     0.217
Recall           0.417     0.700    -0.121    -0.072

=== Results | Campus=campus_10 | Loss=loss3 | TopK=Top 10 ===
                   NCL       SGL    SimGCL   XSimGCL
Hit Ratio        0.563     0.438     0.554    -0.068
NDCG             0.778     0.265     0.372     0.493
Precision        0.567     0.439

In [9]:
import pandas as pd
from colorama import Fore, Style

for model, subdf in df.groupby("Model"):
    print(f"\n===== Model: {model} =====")

    # 先转成字符串表格
    table_str = subdf.to_string(index=False)

    # 按行拆分
    lines = table_str.split("\n")

    # 第一行是表头，原样打印
    print(lines[0])

    # 从第二行开始逐行处理
    for i, (_, row) in enumerate(subdf.iterrows(), start=1):
        line = lines[i]
        if row["RelDiff(%)"] > 0:
            print(Fore.RED + line + Style.RESET_ALL)
        else:
            print(line)



===== Model: NCL =====
Model     Campus  Loss   TopK    Metric  Baseline(loss0)   Value  AbsDiff  RelDiff(%)
  NCL  campus_10 loss1  Top 3 Hit Ratio          0.21944 0.21145 -0.00799   -3.641086
  NCL  campus_10 loss1  Top 3 Precision          0.27139 0.26152 -0.00987   -3.636833
  NCL  campus_10 loss1  Top 3    Recall          0.29537 0.28423 -0.01114   -3.771541
  NCL  campus_10 loss1  Top 3      NDCG          0.35446 0.33997 -0.01449   -4.087908
  NCL  campus_10 loss1  Top 5 Hit Ratio          0.30370 0.29318 -0.01052   -3.463945
  NCL  campus_10 loss1  Top 5 Precision          0.22537 0.21756 -0.00781   -3.465412
  NCL  campus_10 loss1  Top 5    Recall          0.38103 0.37129 -0.00974   -2.556229
  NCL  campus_10 loss1  Top 5      NDCG          0.37450 0.36115 -0.01335   -3.564753
  NCL  campus_10 loss1 Top 10 Hit Ratio          0.43708 0.43051 -0.00657   -1.503157
  NCL  campus_10 loss1 Top 10 Precision          0.16217 0.15973 -0.00244   -1.504594
  NCL  campus_10 loss1 Top 10 